In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.feature_engineering import prepare_features

In [2]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

TARGET_COLUMN = "Цена"

X_train_features = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_test_features = prepare_features(X_test)

y = train[TARGET_COLUMN].copy()

Используем почти тот же набор признаков, что и Ridge:

In [3]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

feature_columns = [
    column
    for column in X_train_features.columns
    if column not in EXCLUDED_COLUMNS
]

X = X_train_features[feature_columns].copy()
X_final_test = X_test_features[feature_columns].copy()

assert list(X.columns) == list(X_final_test.columns)
assert "Предложение" not in X.columns
assert "car_id" not in X.columns

Определяем числовые и категориальные признаки:

In [4]:
numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

categorical_columns = [
    column
    for column in feature_columns
    if column not in numeric_columns
]

print("Числовых признаков:", len(numeric_columns))
print("Категориальных признаков:", len(categorical_columns))
print(categorical_columns)

Числовых признаков: 9
Категориальных признаков: 13
['Бренд', 'Модель', 'Тип машины', 'Полное название', 'Исползование', 'КПП', 'Двигатель', 'Привод', 'Топливо', 'Цвет', 'Локация', 'Тип кузова', 'Штат']


Для CatBoost категориальные пропуски превращаем в отдельную строковую категорию:

In [5]:
for column in categorical_columns:
    X[column] = (
        X[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_final_test[column] = (
        X_final_test[column]
        .fillna("__MISSING__")
        .astype(str)
    )

Создаём тот же stratified split:

In [6]:
target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
)

X_train_split, X_valid_split, y_train_split, y_valid_split = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=target_bins,
)

print(X_train_split.shape)
print(X_valid_split.shape)

(6672, 22)
(1668, 22)


Обучаемся на log1p(Цена):

In [7]:
catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
)

catboost_model.fit(
    X_train_split,
    np.log1p(y_train_split),
    cat_features=categorical_columns,
    eval_set=(
        X_valid_split,
        np.log1p(y_valid_split),
    ),
    use_best_model=True,
    early_stopping_rounds=150,
)

0:	learn: 0.6525135	test: 0.6502147	best: 0.6502147 (0)	total: 195ms	remaining: 4m 52s
100:	learn: 0.2367269	test: 0.2465852	best: 0.2465852 (100)	total: 7.63s	remaining: 1m 45s
200:	learn: 0.2053038	test: 0.2271902	best: 0.2271902 (200)	total: 14.7s	remaining: 1m 35s
300:	learn: 0.1826738	test: 0.2165785	best: 0.2165613 (299)	total: 22.5s	remaining: 1m 29s
400:	learn: 0.1672919	test: 0.2105416	best: 0.2105416 (400)	total: 33.4s	remaining: 1m 31s
500:	learn: 0.1558086	test: 0.2061257	best: 0.2061016 (499)	total: 51s	remaining: 1m 41s
600:	learn: 0.1465576	test: 0.2029779	best: 0.2029779 (600)	total: 1m 8s	remaining: 1m 42s
700:	learn: 0.1389686	test: 0.2010746	best: 0.2010746 (700)	total: 1m 17s	remaining: 1m 28s
800:	learn: 0.1325167	test: 0.1996327	best: 0.1996084 (798)	total: 1m 26s	remaining: 1m 15s
900:	learn: 0.1264974	test: 0.1983249	best: 0.1983249 (900)	total: 1m 34s	remaining: 1m 2s
1000:	learn: 0.1208108	test: 0.1973028	best: 0.1972761 (998)	total: 1m 42s	remaining: 51s
1100

CatBoostRegressor(allow_writing_files=False, depth=8, iterations=1500, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

Оцениваем на исходной шкале цены:

In [8]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


y_pred_train = np.expm1(
    catboost_model.predict(X_train_split)
)

y_pred_valid = np.expm1(
    catboost_model.predict(X_valid_split)
)

y_pred_train = np.maximum(y_pred_train, 1)
y_pred_valid = np.maximum(y_pred_valid, 1)

catboost_result = pd.DataFrame(
    {
        "model": ["catboost_log1p_target"],
        "best_iteration": [catboost_model.get_best_iteration()],
        "train_mape_pct": [
            round(mape_percent(y_train_split, y_pred_train), 3)
        ],
        "validation_mape_pct": [
            round(mape_percent(y_valid_split, y_pred_valid), 3)
        ],
        "train_mae": [
            round(mean_absolute_error(y_train_split, y_pred_train), 2)
        ],
        "validation_mae": [
            round(mean_absolute_error(y_valid_split, y_pred_valid), 2)
        ],
    }
)

display(catboost_result)

,model,best_iteration,train_mape_pct,validation_mape_pct,train_mae,validation_mae
0,catboost_log1p_target,1499,9.337,13.694,4082.03,5020.76


загружаю сохранённый результат Ridge:

In [9]:
REPORTS_DIR = PROJECT_ROOT / "reports"

experiment_results = pd.read_csv(
    REPORTS_DIR / "experiment_results.csv"
)

ridge_result_for_comparison = experiment_results.loc[
    experiment_results["experiment_id"]
    == "ridge_ohe_log1p_alpha_0_1_holdout_rs42"
].copy()

display(ridge_result_for_comparison)

,experiment_id,model,split,train_mape_pct,validation_mape_pct,train_size,validation_size,notes
0,ridge_ohe_log1p_alpha_0_1_holdout_rs42,Ridge + OHE + log1p(target),train_valid_80_20_stratified_qcut10_rs42,2.626,14.194,6672,1668,car_id и Предложение исключены; alpha=0.1


Готовлю результат CatBoost в том же формате:

In [10]:
catboost_result_for_comparison = pd.DataFrame(
    {
        "experiment_id": ["catboost_log1p_holdout_rs42"],
        "model": ["CatBoost + log1p(target)"],
        "split": ["train_valid_80_20_stratified_qcut10_rs42"],
        "train_mape_pct": [
            round(mape_percent(y_train_split, y_pred_train), 3)
        ],
        "validation_mape_pct": [
            round(mape_percent(y_valid_split, y_pred_valid), 3)
        ],
        "train_size": [len(y_train_split)],
        "validation_size": [len(y_valid_split)],
        "notes": [
            f"best_iteration={catboost_model.get_best_iteration()}"
        ],
    }
)

И теперь сравнение:

In [11]:
comparison = pd.concat(
    [
        ridge_result_for_comparison,
        catboost_result_for_comparison,
    ],
    ignore_index=True,
)

comparison = comparison.sort_values(
    "validation_mape_pct"
).reset_index(drop=True)

display(
    comparison[
        [
            "model",
            "train_mape_pct",
            "validation_mape_pct",
            "notes",
        ]
    ]
)

,model,train_mape_pct,validation_mape_pct,notes
0,CatBoost + log1p(target),9.337,13.694,best_iteration=1499
1,Ridge + OHE + log1p(target),2.626,14.194,car_id и Предложение исключены; alpha=0.1


CatBoost — новый лидер, но пока только на одном holdout. Он заслужил полноценную 5-fold CV-проверку, прежде чем тратить вторую попытку leaderboard.

14–16% → устойчиво подтвердить CatBoost

→ улучшить признаки

→ тюнинг CatBoost

→ OOF-анализ и ансамбль

→ только затем думать о 2–3%

Запускаю 5-fold stratified CV для текущего CatBoost-конфига. Это займёт заметно больше времени, чем holdout, но даст честный ориентир.

In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_percentage_error

In [13]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


X_catboost_all = X.copy()

for column in categorical_columns:
    X_catboost_all[column] = (
        X_catboost_all[column]
        .fillna("__MISSING__")
        .astype(str)
    )

cv_target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
).to_numpy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

oof_predictions = pd.Series(
    index=X_catboost_all.index,
    dtype=float,
)

cv_rows = []

for fold_number, (train_idx, valid_idx) in enumerate(
    cv.split(X_catboost_all, cv_target_bins),
    start=1,
):
    X_fold_train = X_catboost_all.iloc[train_idx]
    X_fold_valid = X_catboost_all.iloc[valid_idx]

    y_fold_train = y.iloc[train_idx]
    y_fold_valid = y.iloc[valid_idx]

    fold_model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=1500,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    fold_model.fit(
        X_fold_train,
        np.log1p(y_fold_train),
        cat_features=categorical_columns,
        eval_set=(X_fold_valid, np.log1p(y_fold_valid)),
        use_best_model=True,
        early_stopping_rounds=150,
    )

    fold_pred = np.maximum(
        np.expm1(fold_model.predict(X_fold_valid)),
        1,
    )

    oof_predictions.iloc[valid_idx] = fold_pred

    cv_rows.append(
        {
            "fold": fold_number,
            "best_iteration": fold_model.get_best_iteration(),
            "validation_mape_pct": round(
                mape_percent(y_fold_valid, fold_pred),
                3,
            ),
        }
    )

catboost_cv_folds = pd.DataFrame(cv_rows)

catboost_cv_summary = pd.DataFrame(
    {
        "model": ["CatBoost + log1p(target)"],
        "cv_mape_mean_pct": [
            round(catboost_cv_folds["validation_mape_pct"].mean(), 3)
        ],
        "cv_mape_std_pct": [
            round(catboost_cv_folds["validation_mape_pct"].std(), 3)
        ],
        "oof_mape_pct": [
            round(mape_percent(y, oof_predictions), 3)
        ],
        "mean_best_iteration": [
            round(catboost_cv_folds["best_iteration"].mean(), 1)
        ],
    }
)

display(catboost_cv_folds)
display(catboost_cv_summary)

,fold,best_iteration,validation_mape_pct
0,1,1476,14.535
1,2,1499,12.944
2,3,1497,13.550
3,4,1496,13.826
4,5,1499,13.123


,model,cv_mape_mean_pct,cv_mape_std_pct,oof_mape_pct,mean_best_iteration
0,CatBoost + log1p(target),13.596,0.63,13.596,1493.4


сохраняю OOF-предсказания: они понадобятся для честного ансамбля с Ridge.

In [14]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

catboost_oof = pd.DataFrame(
    {
        "car_id": train.loc[X_catboost_all.index, "car_id"].to_numpy(),
        "y_true": y.to_numpy(),
        "catboost_pred": oof_predictions.to_numpy(),
    }
)

catboost_oof["ape_pct"] = (
    np.abs(catboost_oof["y_true"] - catboost_oof["catboost_pred"])
    / catboost_oof["y_true"]
    * 100
)

catboost_oof.to_parquet(
    REPORTS_DIR / "catboost_oof_predictions.parquet",
    index=False,
)

Для submission обучим CatBoost на всех 8340 объектах. Но перед этим разумно сделать один контролируемый шаг:

iterations: 1500 → 3000

остальные параметры не менять

Не стоит прямо сейчас одновременно менять:

depth;

learning_rate;

l2_leaf_reg;

набор признаков;

функцию потерь;

количество итераций.

Иначе не поймём, что именно дало улучшение.

повторяю только holdout-эксперимент с 3000 итерациями.

In [15]:
catboost_3000 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_3000.fit(
    X_train_split,
    np.log1p(y_train_split),
    cat_features=categorical_columns,
    eval_set=(
        X_valid_split,
        np.log1p(y_valid_split),
    ),
    use_best_model=True,
    early_stopping_rounds=200,
)

y_pred_valid_3000 = np.maximum(
    np.expm1(catboost_3000.predict(X_valid_split)),
    1,
)

catboost_3000_result = pd.DataFrame(
    {
        "model": ["CatBoost + log1p(target), 3000 iterations"],
        "best_iteration": [catboost_3000.get_best_iteration()],
        "validation_mape_pct": [
            round(
                mape_percent(y_valid_split, y_pred_valid_3000),
                3,
            )
        ],
    }
)

display(catboost_3000_result)

0:	learn: 0.6525135	test: 0.6502147	best: 0.6502147 (0)	total: 69.2ms	remaining: 3m 27s
300:	learn: 0.1826738	test: 0.2165785	best: 0.2165613 (299)	total: 22.8s	remaining: 3m 24s
600:	learn: 0.1465576	test: 0.2029779	best: 0.2029779 (600)	total: 46.8s	remaining: 3m 6s
900:	learn: 0.1264974	test: 0.1983249	best: 0.1983249 (900)	total: 1m 11s	remaining: 2m 46s
1200:	learn: 0.1103983	test: 0.1953474	best: 0.1953242 (1199)	total: 1m 35s	remaining: 2m 22s
1500:	learn: 0.0973513	test: 0.1936903	best: 0.1936903 (1500)	total: 1m 59s	remaining: 1m 59s
1800:	learn: 0.0865802	test: 0.1925527	best: 0.1925413 (1796)	total: 2m 24s	remaining: 1m 36s
2100:	learn: 0.0775606	test: 0.1918792	best: 0.1918604 (2082)	total: 2m 48s	remaining: 1m 12s
2400:	learn: 0.0700476	test: 0.1914000	best: 0.1913961 (2398)	total: 3m 13s	remaining: 48.3s
2700:	learn: 0.0632514	test: 0.1912353	best: 0.1911921 (2580)	total: 3m 38s	remaining: 24.2s
2999:	learn: 0.0570065	test: 0.1908425	best: 0.1908425 (2999)	total: 4m 2s	re

,model,best_iteration,validation_mape_pct
0,"CatBoost + log1p(target), 3000 iterations",2999,13.433


In [16]:
catboost_holdout = pd.DataFrame(
    {
        "car_id": train.loc[y_valid_split.index, "car_id"].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "catboost_pred": y_pred_valid_3000,
    }
)

catboost_holdout.to_parquet(
    REPORTS_DIR / "catboost_holdout_predictions_3000.parquet",
    index=False,
)

display(catboost_holdout.head())

,car_id,y_true,catboost_pred
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,27571.576923
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,73718.393132
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,32367.119895
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,17131.541858
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,26695.539841
